# 04 · Leakage-Free Splits

**Reads `labels_consolidated.csv` (nb 03) and `identity_dfuc.csv` (nb 01).**

Builds two patient-grouped 5-fold splits and verifies zero leakage on the image
table. This is the notebook that fixes the failure that started the whole
project: a file-level split that put augmented siblings of 85% of test images
into training while every class-balance check passed.

## The rule
- **Evaluation unit** = the content photograph (hash cluster).
- **Split group** = patients joined by any shared cluster, false positives
  included. If two patients might be linked, they stay on the same side of every
  fold.
- **Conflicts** = a cluster carrying two labels is removed, not guessed.

## Verification is the point
Every check runs on the full image table, not an aggregated one, and the notebook
*asserts* rather than prints. An aggregated table hides patients dropped by
first()-style merges, which is exactly how a leak once passed a check that
reported success. Five keys must all return zero: patient, photograph, cluster,
group, and duplicate paths.

## Two tasks, two splits
- `folds_severity.csv` — Roboflow, one row per consolidated photograph. Note that
  mild is ~3 photographs per fold, so it is effectively unmeasurable; notebook 06
  also reports a collapsed severe-vs-rest result.
- `folds_infection.csv` — DFUC expert infection labels, the control experiment.

## Outputs
`folds_severity.csv`, `folds_infection.csv`


In [10]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np

INTERIM = Path('/Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/interim')
SEV_LABELS  = INTERIM / 'labels_consolidated.csv'   # from notebook 03
DFUC_IDENT  = INTERIM / 'identity_dfuc.csv'         # from notebook 01

N_FOLDS, SEED = 5, 42

missing = [str(p) for p in [SEV_LABELS, DFUC_IDENT] if not p.exists()]
if missing:
    print('STOPPING. Missing inputs:')
    for m in missing: print('  -', m)
    print('Run notebooks 01 and 03 first.')
    raise SystemExit(1)
print('inputs found. building two leakage-free splits: severity and infection.')

inputs found. building two leakage-free splits: severity and infection.


In [11]:
# Cell 2 · shared tools: grouping and leakage verification
from sklearn.model_selection import StratifiedGroupKFold

class DSU:
    def __init__(s, keys):
        s.p = {k: k for k in keys}
    def find(s, x):
        while s.p[x] != x:
            s.p[x] = s.p[s.p[x]]; x = s.p[x]
        return x
    def union(s, a, b):
        ra, rb = s.find(a), s.find(b)
        if ra != rb: s.p[rb] = ra

def link_patients_by_cluster(df):
    # patients that share a content cluster go in the same split group,
    # even if the link is a false positive: being wrong about a merge is
    # cheap, leaking is not.
    dsu = DSU(df.patient_id.unique())
    for _, g in df.groupby('photo_unit'):
        pats = g.patient_id.unique()
        for p in pats[1:]:
            dsu.union(pats[0], p)
    return df.patient_id.map(dsu.find)

def verify_no_leakage(img_df, keys):
    # runs on the IMAGE table, not an aggregated one. An aggregated table
    # hides patients dropped by first()-style merges, which is exactly how
    # a leak once passed a check that reported success.
    ok = True
    for key in keys:
        spans = img_df.groupby(key)['fold'].nunique()
        n = int((spans > 1).sum())
        print(f'    {key:<14} in >1 fold: {n}')
        ok &= (n == 0)
    dup = int(img_df.path.duplicated().sum())
    print(f'    {"duplicate paths":<14}         : {dup}')
    return ok and dup == 0

print('grouping and verification tools ready')

grouping and verification tools ready


In [12]:
# Cell 3 · severity folds (Roboflow, consolidated)
# labels_consolidated.csv is already one row per photograph with a patient.
# Group by patient component, stratify on the severity label.
sev = pd.read_csv(SEV_LABELS)
sev['group'] = link_patients_by_cluster(sev)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
sev['fold'] = -1
for k, (_, te) in enumerate(sgkf.split(sev, sev.y, groups=sev.group)):
    sev.iloc[te, sev.columns.get_loc('fold')] = k
assert (sev.fold >= 0).all()

print('severity folds (photograph level):')
print(f"  {'fold':<6}{'photos':>8}{'mild':>7}{'mod':>7}{'sev':>7}")
for k in range(N_FOLDS):
    f = sev[sev.fold == k]
    n = f.severity.value_counts()
    print(f"  {k:<6}{len(f):>8}{n.get('mild',0):>7}"
          f"{n.get('moderate',0):>7}{n.get('severe',0):>7}")

# note the mild reality up front
mild_per_fold = [int((sev[sev.fold==k].severity=='mild').sum()) for k in range(N_FOLDS)]
print(f'\n  mild photographs per fold: {mild_per_fold}')
if max(mild_per_fold) < 8:
    print('  mild is effectively unmeasurable at photograph level.')
    print('  notebook 06 will also report a collapsed severe-vs-rest result.')

severity folds (photograph level):
  fold    photos   mild    mod    sev
  0          723      6    247    470
  1          723      6    247    470
  2          723      6    247    470
  3          723      6    247    470
  4          722      6    247    469

  mild photographs per fold: [6, 6, 6, 6, 6]
  mild is effectively unmeasurable at photograph level.
  notebook 06 will also report a collapsed severe-vs-rest result.


In [13]:
# Cell 4 · infection folds (DFUC, expert labels)
# Derive the infection label from the folder name, remove label conflicts,
# then group and fold. This is the control-experiment split.
dfuc = pd.read_csv(DFUC_IDENT)

def infection_label(path):
    p = str(path).lower()
    if 'not_infected' in p or 'non_infected' in p: return 0
    if 'infected' in p:                             return 1
    return np.nan   # ischaemia-axis images fall through

dfuc['label'] = dfuc.path.map(infection_label)
dfuc = dfuc[dfuc.label.notna()].copy()
dfuc['label'] = dfuc.label.astype(int)
print(f'infection-axis images: {len(dfuc):,}')

# evaluation unit = content cluster. Remove clusters carrying both labels.
k = dfuc.groupby('photo_unit').label.nunique()
conflict = set(k[k > 1].index)
print(f'  conflict clusters (both labels): {len(conflict)}  '
      f'-> {int(dfuc.photo_unit.isin(conflict).sum())} images removed')
dfuc = dfuc[~dfuc.photo_unit.isin(conflict)].copy()

dfuc['group'] = link_patients_by_cluster(dfuc)

# one row per unit for fold assignment, stratified on label
unit = dfuc.groupby('photo_unit').agg(
    label=('label', 'first'), group=('group', 'first')).reset_index()
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
unit['fold'] = -1
for kk, (_, te) in enumerate(sgkf.split(unit, unit.label, groups=unit.group)):
    unit.iloc[te, unit.columns.get_loc('fold')] = kk
dfuc = dfuc.merge(unit[['photo_unit', 'fold']], on='photo_unit')

print(f'\n  evaluation units: {unit.photo_unit.nunique():,}')
print(f'  patients: {dfuc.patient_id.nunique():,}  '
      f'-> split groups: {dfuc.group.nunique():,}')
print(f"  {'fold':<6}{'units':>8}{'pos':>7}{'neg':>7}")
for kk in range(N_FOLDS):
    u = unit[unit.fold == kk]
    print(f"  {kk:<6}{len(u):>8}{int((u.label==1).sum()):>7}"
          f"{int((u.label==0).sum()):>7}")

infection-axis images: 5,810
  conflict clusters (both labels): 19  -> 438 images removed

  evaluation units: 4,542
  patients: 1,417  -> split groups: 1,390
  fold     units    pos    neg
  0          909    550    359
  1          909    550    359
  2          908    550    358
  3          908    550    358
  4          908    549    359


In [14]:
# Cell 5 · leakage verification — the point of this notebook
# Both splits are checked on the full image table. Every key must return
# zero. The notebook asserts, so a leak stops the pipeline here rather
# than silently propagating into training.
print('SEVERITY split verification (on all Roboflow images):')
# expand consolidated photographs back to every image via labels_raw
raw = pd.read_csv(INTERIM / 'labels_raw.csv')
sev_img = raw.merge(sev[['photo_unit', 'fold']], on='photo_unit', how='inner')
sev_ok = verify_no_leakage(sev_img, ['patient_id', 'photo_id', 'hash_cluster', 'photo_unit'])

print('\nINFECTION split verification (on all DFUC infection images):')
inf_ok = verify_no_leakage(dfuc, ['patient_id', 'photo_id', 'hash_cluster', 'photo_unit', 'group'])

assert sev_ok, 'LEAKAGE in severity split'
assert inf_ok, 'LEAKAGE in infection split'
print('\nboth splits verified: nothing spans a fold.')

SEVERITY split verification (on all Roboflow images):
    patient_id     in >1 fold: 0
    photo_id       in >1 fold: 0
    hash_cluster   in >1 fold: 0
    photo_unit     in >1 fold: 0
    duplicate paths         : 0

INFECTION split verification (on all DFUC infection images):
    patient_id     in >1 fold: 0
    photo_id       in >1 fold: 0
    hash_cluster   in >1 fold: 0
    photo_unit     in >1 fold: 0
    group          in >1 fold: 0
    duplicate paths         : 0

both splits verified: nothing spans a fold.


In [15]:
# Cell 6 · save both splits
sev_out = sev[['photo_unit', 'hash_cluster', 'photo_id', 'patient_id', 'group',
               'severity', 'y', 'fold', 'representative']]
sev_out.to_csv(INTERIM / 'folds_severity.csv', index=False)
print(f'wrote {(INTERIM / "folds_severity.csv").resolve()}  '
      f'({len(sev_out):,} photographs)')

inf_out = dfuc[['path', 'photo_unit', 'hash_cluster', 'photo_id', 'patient_id',
                'group', 'label', 'fold']]
inf_out.to_csv(INTERIM / 'folds_infection.csv', index=False)
print(f'wrote {(INTERIM / "folds_infection.csv").resolve()}  '
      f'({len(inf_out):,} images, {inf_out.hash_cluster.nunique():,} units)')

wrote /Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/interim/folds_severity.csv  (3,614 photographs)
wrote /Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/interim/folds_infection.csv  (5,372 images, 4,542 units)


In [16]:
# Cell 7 · summary
print('=' * 58)
print('STAGE 04 COMPLETE')
print('=' * 58)
print(f'  folds_severity.csv  : {len(sev):,} photographs, '
      f'{sev.group.nunique():,} groups')
print(f'  folds_infection.csv : {dfuc.hash_cluster.nunique():,} units, '
      f'{dfuc.group.nunique():,} groups')
print(f'  folds               : {N_FOLDS}, grouped by patient component')
print(f'  leakage             : zero across all keys, both splits')
print('\n  train on fold != k, test on fold == k, with confidence.')
print('\nnext: 05_feature_extraction.ipynb  (heavy compute, has a Colab twin)')

STAGE 04 COMPLETE
  folds_severity.csv  : 3,614 photographs, 3,614 groups
  folds_infection.csv : 4,542 units, 1,390 groups
  folds               : 5, grouped by patient component
  leakage             : zero across all keys, both splits

  train on fold != k, test on fold == k, with confidence.

next: 05_feature_extraction.ipynb  (heavy compute, has a Colab twin)
